# Second 1.0 — Phase 8: Browser Model Export & GGUF Quantization

This notebook executes **Phase 8** from `PLAN.md`:
1. **Locate Winning Checkpoint**: Loads `Qwen/Qwen3.5-2B` (`bfloat16`) and merges the winning Stage T4 LoRA adapter (`RL-B2`, `r=32`, `alpha=64`) using `PeftModel.merge_and_unload()`.
2. **Strip Vision Tower**: Strips visual modules via `strip_vision_parameters` to ensure clean, compact text causal LM weights.
3. **Convert to GGUF (Q4_K_M)**: Uses `llama.cpp` (`convert_hf_to_gguf.py` + `llama-quantize`) to create `Second-1.0-2B-Q4_K_M.gguf` for `wllama` WebAssembly.
4. **Shard for Browser Caching**: Partitions GGUF with `llama-gguf-split` into `< 500 MB` shards (`-00001-of-00003.gguf`, etc.) for Chrome Side Panel execution.
5. **Batch GGUF Inference**: Executes greedy decoding (`temp=0`) on the 100 stratified test prompts using `export/generate_gguf_completions.py`.
6. **Post-Quantization Quality Check**: Runs all 9 deterministic rule verifiers against `eval/results/grpo-1.0-2b-v2_outputs.jsonl` to verify hard-fail rate is $\le 6.70\%$.
7. **Publish to Hugging Face**: Uploads to `second-chess/Second-1.0-2B` and `second-chess/Second-1.0-2B-GGUF`.
8. **Sync to Drive**: Backs up GGUFs and evaluation reports to Google Drive.

In [ ]:
# Step 1: Environment Check, Mount Google Drive & Unpack Bundle
from pathlib import Path
import os, sys, zipfile
import torch

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (sufficient for GGUF conversion & quantization).")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Mounted Google Drive successfully.")
except ImportError:
    print("Not running in Google Colab. Using local filesystem.")

WORK_DIR = Path("/content/Second")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    Path("/content/drive/MyDrive/second/phase8_export_bundle.zip"),
    Path("/content/drive/MyDrive/second/data/phase8_export_bundle.zip"),
    Path("/content/phase8_export_bundle.zip"),
    Path("data/processed/phase8_export_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    os.chdir(str(WORK_DIR))
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    print(f"Bundle extracted. Working directory: {os.getcwd()}")
elif Path("export/merge_lora.py").is_file():
    print(f"Working in existing repository at {os.getcwd()}")
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
else:
    print("Note: Upload phase8_export_bundle.zip to /content or Google Drive before running Step 4.")

ADAPTER_PATH = Path("/content/drive/MyDrive/second/checkpoints/grpo_v2_b2/final_adapter")
if ADAPTER_PATH.is_dir():
    print(f"Found winning RL-B2 adapter at: {ADAPTER_PATH}")
else:
    print(f"Note: Adapter path {ADAPTER_PATH} not yet mounted or not found.")


In [ ]:
# Step 2: Install Packages & Apply TorchAO/Torchaudio Compatibility Workaround
# Uninstall conflicting multimedia extensions that cause libc10_cuda.so errors in Colab
!pip uninstall -y -q torchaudio torchvision torchao 2>/dev/null || true
# Install latest transformers from GitHub main to ensure Qwen 3.5 architecture support
!pip install -q "git+https://github.com/huggingface/transformers.git" peft accelerate safetensors huggingface_hub python-chess pyarrow gguf sentencepiece protobuf

import sys
sys.modules["torchaudio"] = None

# Apply peft and transformers workarounds
try:
    import transformers.utils.import_utils
    transformers.utils.import_utils.is_torchaudio_available = lambda: False
except Exception:
    pass

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
print("Installed packages and configured PEFT/Transformers environment.")

In [ ]:
# Step 3: Clone and Build llama.cpp for GGUF Tools
%cd /content
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
%cd /content/llama.cpp
# Install conversion tools without overwriting latest transformers
!pip install -q gguf sentencepiece protobuf
import torch
cuda_flag = "-DGGML_CUDA=ON" if torch.cuda.is_available() else "-DGGML_CUDA=OFF"
print(f"Configuring llama.cpp build with {cuda_flag}...")
!cmake -B build {cuda_flag}
!cmake --build build --config Release -j --target llama-quantize llama-gguf-split llama-completion llama-cli

assert Path("/content/llama.cpp/convert_hf_to_gguf.py").is_file(), "convert_hf_to_gguf.py missing!"
assert Path("/content/llama.cpp/build/bin/llama-quantize").is_file(), "llama-quantize binary missing!"
assert Path("/content/llama.cpp/build/bin/llama-gguf-split").is_file(), "llama-gguf-split binary missing!"
print("llama.cpp conversion tools built successfully.")

# Return to repository working directory to avoid directory desynchronization
repo_root = Path("/content/Second") if Path("/content/Second/export").is_dir() else Path("/content")
os.chdir(str(repo_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Current working directory restored to: {os.getcwd()}")

In [ ]:
# Step 4: Merge LoRA Adapter with Base Model & Strip Vision Tower
from pathlib import Path
import os, sys
import torch

repo_root = Path("/content/Second") if Path("/content/Second/export").is_dir() else Path.cwd()
os.chdir(str(repo_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from export.merge_lora import merge_and_export_lora

BASE_MODEL_ID = "Qwen/Qwen3.5-2B"
MERGED_DIR = Path("/content/merged_model")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print("Executing merge_and_export_lora with vision tower stripping...")
merge_and_export_lora(
    base_model_path=BASE_MODEL_ID,
    adapter_dir="/content/drive/MyDrive/second/checkpoints/grpo_v2_b2/final_adapter",
    output_dir=MERGED_DIR,
    strip_vision=True,
    torch_dtype="bfloat16",
    device_map="auto" if torch.cuda.is_available() else "cpu",
)
print("Merged text causal LM model saved successfully.")


In [ ]:
# Step 5: Convert to GGUF, Quantize to Q4_K_M, and Shard for Browser (< 500 MB)
GGUF_DIR = Path("/content/gguf")
GGUF_DIR.mkdir(parents=True, exist_ok=True)

F16_GGUF = GGUF_DIR / "Second-1.0-2B-f16.gguf"
Q4_GGUF = GGUF_DIR / "Second-1.0-2B-Q4_K_M.gguf"

# 1. Convert to FP16 GGUF
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model --outfile {F16_GGUF} --outtype f16

# 2. Quantize to Q4_K_M
!/content/llama.cpp/build/bin/llama-quantize {F16_GGUF} {Q4_GGUF} Q4_K_M

# 3. Shard into < 500 MB pieces for browser caching using llama-gguf-split
!/content/llama.cpp/build/bin/llama-gguf-split --split-max-size 480M {Q4_GGUF} {GGUF_DIR / "Second-1.0-2B-Q4_K_M"}

# Remove temporary f16 file to free disk
if F16_GGUF.exists():
    F16_GGUF.unlink()

print("GGUF files created:")
!ls -lh /content/gguf/

In [ ]:
# Step 6: Batch Inference on 100 Test Prompts & Quality Check
import os, sys, re, json, subprocess
from pathlib import Path

# Build llama-completion in the existing Colab session if not yet built
LLAMA_COMPLETION_BIN = Path("/content/llama.cpp/build/bin/llama-completion")
if not LLAMA_COMPLETION_BIN.is_file() and Path("/content/llama.cpp/build").is_dir():
    print("Building llama-completion target (llama-cli in current llama.cpp is interactive chat only and rejects -no-cnv)...")
    subprocess.run(
        ["cmake", "--build", "/content/llama.cpp/build", "--config", "Release", "-j", "--target", "llama-completion"],
        check=True,
    )

repo_root = Path("/content/Second") if Path("/content/Second/export").is_dir() else Path.cwd()
os.chdir(str(repo_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from eval.baselines import load_jsonl
from export.generate_gguf_completions import generate_gguf_batch_completions
from export.quality_check import select_100_test_prompts, run_quality_check_and_save

def extract_clean_completion(raw_stdout: str) -> str:
    """Strip ANSI escapes, prompt echoes, think blocks, and timings from llama-completion output."""
    ansi_escape = re.compile(r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])")
    text = ansi_escape.sub("", raw_stdout)
    if "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1]
    for stop_token in ("<|im_end|>", "<|endoftext|>", "[end of text]"):
        if stop_token in text:
            text = text.split(stop_token)[0]
    text = re.sub(r"^\s*<think>.*?</think>\s*", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = text.replace("<think>", "").replace("</think>", "")
    lines = [
        line for line in text.splitlines()
        if not any(line.strip().startswith(p) for p in [
            "llama_print_timings:", "llama_perf_", "common_perf_print:",
            "load time =", "prompt eval time =", "eval time =", "total time =",
            "system_info:", "sampling time =", "sampler "
        ])
    ]
    return "\n".join(lines).strip()

GGUF_OUTPUTS = Path("eval/results/export_gguf_q4_k_m_outputs.jsonl")
MODEL_PATH = Path("/content/gguf/Second-1.0-2B-Q4_K_M.gguf")

testset = load_jsonl("eval/testset-1.0.jsonl")
items = select_100_test_prompts(testset, strategy="stratified")[:100]

print(f"Generating completions for {len(items)} prompts with llama-completion...")
results = []
for idx, it in enumerate(items):
    rec = dict(it)
    prompt = it.get("chat_prompt") or it.get("user_message", "")
    cmd = [
        str(LLAMA_COMPLETION_BIN), "-m", str(MODEL_PATH), "-p", prompt,
        "-n", "200", "-c", "1024", "--temp", "0.0", "-r", "<|im_end|>",
        "--no-display-prompt", "--simple-io", "--color", "off", "-no-cnv", "-ngl", "99"
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if proc.returncode != 0:
        raise RuntimeError(f"llama-completion failed on prompt {idx} (exit {proc.returncode}):\n{proc.stderr}")
    rec["completion"] = extract_clean_completion(proc.stdout)
    if idx == 0:
        print(f"Sample completion [0]: {rec['completion'][:160]}...")
    rec["output"] = rec["completion"]
    rec["model"] = "Second-1.0-2B-GGUF-Q4_K_M"
    results.append(rec)

GGUF_OUTPUTS.parent.mkdir(parents=True, exist_ok=True)
with open(GGUF_OUTPUTS, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print("Evaluating quality check against release gates...")
res = run_quality_check_and_save(
    testset_path="eval/testset-1.0.jsonl",
    canonical_outputs_path="eval/results/grpo-1.0-2b-v2_outputs.jsonl",
    quant_outputs_path=GGUF_OUTPUTS,
    eval_json_path="eval/results/export_gguf_eval.json",
    report_path="eval/reports/export_quantization_report.md",
    strategy="stratified",
    quant_type="Q4_K_M",
)

print(f"Post-quantization hard-fail rate: {res.quant_hard_fail_pct:.2f}% (Release gate <= 6.70%)")
print(f"Gate verification status: {res.recommendation}")
assert res.passed_hard_fail_gate, f"Quality gate failed! Hard fail rate: {res.quant_hard_fail_pct:.2f}%"


In [ ]:
# Step 7: Publish Models and Model Cards to Hugging Face Hub
import os, sys
from pathlib import Path
from huggingface_hub import HfApi

repo_root = Path("/content/Second") if Path("/content/Second/export").is_dir() else Path.cwd()
os.chdir(str(repo_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from export.publish_hf import publish_all_phase8_models

# Always reload HF_TOKEN from Colab secrets if available (in case an older read-only token was cached in os.environ)
try:
    from google.colab import userdata
    secret_tok = userdata.get("HF_TOKEN")
    if secret_tok and secret_tok.strip():
        os.environ["HF_TOKEN"] = secret_tok.strip()
except Exception:
    pass

hf_token = os.environ.get("HF_TOKEN")
assert hf_token, "Set HF_TOKEN in environment or Colab secrets before uploading!"

api = HfApi(token=hf_token)
who = api.whoami()
username = who.get("name")
org_names = [o.get("name") for o in who.get("orgs", [])]
role = who.get("auth", {}).get("accessToken", {}).get("role", "unknown")
print(f"Authenticated as: {username} (token role: {role}, orgs: {org_names})")

# Use second-chess org if your account/token has access to it; otherwise publish under your HF username
target_namespace = "second-chess" if ("second-chess" in org_names or username == "second-chess") else username
if target_namespace != "second-chess":
    print(f"Note: Organization 'second-chess' not in {org_names}. Publishing under '{target_namespace}/' instead.")

ADAPTER_DIR = Path("/content/drive/MyDrive/second/checkpoints/grpo_v2_b2/final_adapter")
upload_res = publish_all_phase8_models(
    merged_dir="/content/merged_model",
    gguf_dir="/content/gguf",
    adapter_dir=str(ADAPTER_DIR) if ADAPTER_DIR.is_dir() else None,
    base_repo_id=f"{target_namespace}/Second-1.0-2B",
    gguf_repo_id=f"{target_namespace}/Second-1.0-2B-GGUF",
    model_card_base="export/model_cards/Second-1.0-2B/README.md",
    model_card_gguf="export/model_cards/Second-1.0-2B-GGUF/README.md",
    token=hf_token,
)
print("Hugging Face publication succeeded!")
print(f"Safetensors: {upload_res['base_model'].get('url')}")
print(f"GGUF: {upload_res['gguf_model'].get('url')}")


In [ ]:
# Step 8: Sync Export Artifacts to Google Drive
import shutil
DRIVE_EXPORT = Path("/content/drive/MyDrive/second/export")
DRIVE_EXPORT.mkdir(parents=True, exist_ok=True)

for f in Path("/content/gguf").glob("*.gguf"):
    print(f"Backing up {f.name} to Drive...")
    shutil.copy2(f, DRIVE_EXPORT / f.name)

shutil.copy2("eval/reports/export_quantization_report.md", DRIVE_EXPORT / "export_quantization_report.md")
shutil.copy2("eval/results/export_gguf_eval.json", DRIVE_EXPORT / "export_gguf_eval.json")
shutil.copy2("eval/results/export_gguf_q4_k_m_outputs.jsonl", DRIVE_EXPORT / "export_gguf_q4_k_m_outputs.jsonl")
print("All Phase 8 artifacts backed up to Google Drive successfully!")